In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def lattice_spacing_su2(beta: float) -> float:
    """SU(2) lattice spacing from arXiv:1811.02800."""
    t0_phys = 0.01133  # fm^2
    delta_beta = beta - 2.600
    ln_t0_over_a2 = 1.285 + 6.409 * delta_beta - 0.7411 * delta_beta**2
    t0_over_a2 = np.exp(ln_t0_over_a2)
    return np.sqrt(t0_phys / t0_over_a2)


def lattice_spacing_su3(beta: float) -> float:
    """SU(3) lattice spacing from arXiv:hep-lat/0108008."""
    r0 = 0.5  # fm
    delta_beta = beta - 6.0
    ln_a_over_r0 = -1.6804 - 1.7331 * delta_beta + 0.7849 * delta_beta**2 - 0.4428 * delta_beta**3
    return np.exp(ln_a_over_r0) * r0


In [4]:
# ---------------------------------------------------------------------------
# Volume-preserving lattice adjustment
# ---------------------------------------------------------------------------

def adjust_lattice(T_ref: int, L_ref: int, a_ref_fm: float, a_new_fm: float,
                   open_bc: bool = False, n_exclude: int = 2) -> dict:
    """Compute (T_new, L_new) keeping physical volume constant.

    open_bc: n_exclude slices discarded per temporal boundary,
             physical temporal extent = (T - 2*n_exclude) * a.
    """
    ratio = a_ref_fm / a_new_fm
    L_new = round(L_ref * ratio)

    if open_bc:
        T_eff_ref = T_ref - 2 * n_exclude
        T_new = round(T_eff_ref * ratio) + 2 * n_exclude
        T_eff_new = T_new - 2 * n_exclude
    else:
        T_new = round(T_ref * ratio)
        T_eff_new = T_new

    V_fm4 = T_eff_new * L_new**3 * a_new_fm**4
    return {"T_new": T_new, "L_new": L_new, "T_eff_new": T_eff_new, "V_fm4": V_fm4}


In [5]:
# ---------------------------------------------------------------------------
# Scan: beta=2.5 upward, reference T=16^3 x 16, open BC
# ---------------------------------------------------------------------------

T_ref, L_ref = 17, 13
beta_ref  = 2.5
a_ref_fm  = lattice_spacing_su2(beta_ref)
open_bc   = True
n_exclude = 2   # must match exclude_boundary_slices in your input file

beta_values = np.arange(2.5, 3.15, 0.05)

print(f"Reference: beta={beta_ref}, T={T_ref}, L={L_ref}, a={a_ref_fm:.4f} fm, open_bc={open_bc}")
print()
print(f"{'beta':>8} {'a [fm]':>10} {'T_new':>7} {'L_new':>7} {'T_eff':>7} {'V [fm^4]':>12}")
print("-" * 55)

for beta in beta_values:
    a = lattice_spacing_su2(beta)
    adj = adjust_lattice(T_ref, L_ref, a_ref_fm, a, open_bc=open_bc, n_exclude=n_exclude)
    print(f"{beta:8.2f} {a:10.4f} {adj['T_new']:7d} {adj['L_new']:7d} "
          f"{adj['T_eff_new']:7d} {adj['V_fm4']:12.6f}")


Reference: beta=2.5, T=17, L=13, a=0.0774 fm, open_bc=True

    beta     a [fm]   T_new   L_new   T_eff     V [fm^4]
-------------------------------------------------------
    2.50     0.0774      17      13      13     1.026153
    2.55     0.0658      19      15      15     0.947625
    2.60     0.0560      22      18      18     1.031367
    2.65     0.0477      25      21      21     1.010346
    2.70     0.0408      29      25      25     1.081039
    2.75     0.0349      33      29      29     1.050462
    2.80     0.0299      38      34      34     1.073074
    2.85     0.0257      43      39      39     1.011851
    2.90     0.0221      49      45      45     0.984172
    2.95     0.0191      57      53      53     1.046901
    3.00     0.0165      65      61      61     1.023108
    3.05     0.0143      75      71      71     1.053547
    3.10     0.0124      85      81      81     1.008783


In [6]:
# ---------------------------------------------------------------------------
# Scan: beta=6.0 upward, reference T=16^3 x 16, open BC  [SU(3)]
# ---------------------------------------------------------------------------

T_ref, L_ref = 18, 14
beta_ref  = 6.1
a_ref_fm  = lattice_spacing_su3(beta_ref)
open_bc   = True
n_exclude = 2   # must match exclude_boundary_slices in your input file

beta_values = np.arange(6.1, 7.45, 0.05)

print(f"Reference: beta={beta_ref}, T={T_ref}, L={L_ref}, a={a_ref_fm:.4f} fm, open_bc={open_bc}")
print()
print(f"{'beta':>8} {'a [fm]':>10} {'T_new':>7} {'L_new':>7} {'T_eff':>7} {'V [fm^4]':>12}")
print("-" * 55)

for beta in beta_values:
    a = lattice_spacing_su3(beta)
    adj = adjust_lattice(T_ref, L_ref, a_ref_fm, a, open_bc=open_bc, n_exclude=n_exclude)
    print(f"{beta:8.2f} {a:10.4f} {adj['T_new']:7d} {adj['L_new']:7d} "
          f"{adj['T_eff_new']:7d} {adj['V_fm4']:12.6f}")

Reference: beta=6.1, T=18, L=14, a=0.0789 fm, open_bc=True

    beta     a [fm]   T_new   L_new   T_eff     V [fm^4]
-------------------------------------------------------
    6.10     0.0789      18      14      14     1.489478
    6.15     0.0730      19      15      15     1.437370
    6.20     0.0677      20      16      16     1.378644
    6.25     0.0630      22      18      18     1.653259
    6.30     0.0587      23      19      19     1.550518
    6.35     0.0549      24      20      20     1.449272
    6.40     0.0513      26      22      22     1.625800
    6.45     0.0481      27      23      23     1.495673
    6.50     0.0451      29      25      25     1.613837
    6.55     0.0423      30      26      26     1.462903
    6.60     0.0397      32      28      28     1.526330
    6.65     0.0373      34      30      30     1.559875
    6.70     0.0349      36      32      32     1.563548
    6.75     0.0328      38      34      34     1.538416
    6.80     0.0307      40  

In [7]:
# ---------------------------------------------------------------------------
# Minimum lattice size for a target spread of Q
# ---------------------------------------------------------------------------

HBAR_C = 197.3  # MeV·fm

X_ref_su2 = 200.0  # MeV, χ_t^(1/4)
X_ref_su3 = 191.0  # MeV, χ_t^(1/4)

spread = 2  # target sqrt(<Q²>)
Q2_target = spread**2

for label, X_ref, lattice_spacing, beta in [
    ("SU(2)", X_ref_su2, lattice_spacing_su2, 2.5),
    ("SU(3)", X_ref_su3, lattice_spacing_su3, 6.10),
]:
    a_fm = lattice_spacing(beta)
    chi_t_fm4 = (X_ref / HBAR_C)**4      # χ_t in fm⁻⁴
    V_phys = Q2_target / chi_t_fm4        # required physical volume in fm⁴
    N_sites = V_phys / a_fm**4            # total lattice sites
    L_sites = N_sites ** (1/4)            # linear extent (hypercubic)

    print(f"{label}  beta={beta}, a={a_fm:.4f} fm")
    print(f"  Required V  = {V_phys:.2f} fm⁴")
    print(f"  Total sites = {N_sites:.2e}")
    print(f"  Linear size = {L_sites:.1f} sites  (hypercubic T=L)")
    print()

SU(2)  beta=2.5, a=0.0774 fm
  Required V  = 3.79 fm⁴
  Total sites = 1.05e+05
  Linear size = 18.0 sites  (hypercubic T=L)

SU(3)  beta=6.1, a=0.0789 fm
  Required V  = 4.55 fm⁴
  Total sites = 1.17e+05
  Linear size = 18.5 sites  (hypercubic T=L)



In [8]:
beta = 3.0
N = 1

a_fm   = lattice_spacing_su2(beta)
X_ref  = X_ref_su2
HBAR_C = 200

chi_t_fm4 = (X_ref / HBAR_C)**4
V         = (N * a_fm)**4

spread = np.sqrt(chi_t_fm4 * V)
print(f"a = {a_fm:.4f} fm,  V = {V:.4f} fm⁴,  spread = {spread:.2f}")


a = 0.0165 fm,  V = 0.0000 fm⁴,  spread = 0.00
